# Batch Cadansanalyse – Dataset C (Puurs)

Automatische extractie van de stappen per minuut (SPM) voor lopers op basis van de opgeslagen AI-detecties.

> **Vereisten:**
> - Detectiebestand `P_J16.json`
>
> **Output:** Statistische samenvatting en SPM Histogram plot.

### Cadansanalyse

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt, find_peaks

# --- INSTELLINGEN ---
JSON_PATH = '../../data/detecties/P_J16.json'
MIN_TRACK_SECONDS = 1.5

print(f"Start met het laden van Y-Sway data uit: {JSON_PATH}...")

with open(JSON_PATH, 'r') as f:
    export_data = json.load(f)

fps = export_data["fps"]
runner_data = {}

for t_id_str, track_info in export_data["tracks"].items():
    t_id = int(t_id_str)
    frames = track_info["frames"]
    y_coords = track_info["y_coords"]

    time_stamps = [frame / fps for frame in frames]

    runner_data[t_id] = {
        'y_coords': y_coords,
        'time_stamps': time_stamps
    }

low_cut = 0.8
high_cut = 2.0
nyquist = 0.5 * fps
b, a = butter(2, [low_cut / nyquist, high_cut / nyquist], btype='band', analog=False)

valid_spms = []

for t_id in sorted(runner_data.keys()):
    y_coords = runner_data[t_id]['y_coords']
    time_stamps = runner_data[t_id]['time_stamps']
    total_track_time = time_stamps[-1] - time_stamps[0]

    if total_track_time < MIN_TRACK_SECONDS or len(y_coords) <= 15:
        continue

    try:
        y_smoothed = filtfilt(b, a, y_coords)
        peaks, _ = find_peaks(y_smoothed, distance=fps*0.4)

        if len(peaks) > 1:
            peak_to_peak_time = time_stamps[peaks[-1]] - time_stamps[peaks[0]]

            if peak_to_peak_time > 0:
                aantal_stappen = (len(peaks) - 1) * 2
                spm = (aantal_stappen / peak_to_peak_time) * 60

                if 80 <= spm <= 240:
                    valid_spms.append(spm)

    except ValueError:
        continue

print("\n" + "="*60)
print("RESULTATEN: BIOMECHANISCHE STAPFREQUENTIE (SPM)")
print("="*60)

if len(valid_spms) > 0:
    avg_spm = sum(valid_spms) / len(valid_spms)
    median_spm = np.median(valid_spms)
    print(f"Gemiddelde Cadans     : {avg_spm:.0f} SPM")
    print(f"Mediaan Cadans        : {median_spm:.0f} SPM")
    print(f"Aantal Lopers Gevat   : {len(valid_spms)}")
else:
    print("Geen lopers gevonden met een betrouwbare stapfrequentie.")
print("="*60)

### Cadanshistogram

In [ ]:
if len(valid_spms) > 0:
    spm_array = np.array(valid_spms)
    gemiddelde_spm = np.mean(spm_array)
    mediaan_spm = np.median(spm_array)

    plt.figure(figsize=(14, 6.5))
    n, bins, patches = plt.hist(spm_array, bins=25, color='#2ca02c', edgecolor='white',
                                linewidth=1.1, alpha=0.85, zorder=3)

    plt.axvline(mediaan_spm, color='#d62728', linewidth=2.8, linestyle='-', label=f'Mediaan ({mediaan_spm:.0f} SPM)', zorder=4)
    plt.axvline(gemiddelde_spm, color='#ff7f0e', linewidth=2.5, linestyle='--', label=f'Gemiddelde ({gemiddelde_spm:.0f} SPM)', zorder=4)

    plt.title("Cadansverdeling (SPM) Dataset C", fontsize=16, fontweight='bold', pad=15)
    plt.xlabel("Stappen Per Minuut (SPM)", fontsize=13, fontweight='bold')
    plt.ylabel("Aantal lopers", fontsize=13, fontweight='bold')
    plt.legend(fontsize=12, loc='upper right')
    plt.grid(True, linestyle='--', alpha=0.6, zorder=0)
    plt.tight_layout()
    plt.show()